In [0]:
%pip install faker

dbutils.library.restartPython()


In [0]:
import os
import random  
from pyspark.sql.functions import udf, current_timestamp
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField
from faker import Faker

driver_schema = StructType([
    StructField("id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("car_number", StringType(), True),
    StructField("experience", IntegerType(), True),
    StructField("rating", DoubleType(), True)
])


def generate_driver_raw():
    fake_worker = Faker()
    return {
        "id": fake_worker.uuid4(),
        "name": fake_worker.name(),
        "car_number": fake_worker.license_plate(),
        "experience": int(fake_worker.random_int(min=1, max=20)),
        "rating": float(round(random.uniform(3.5, 5.0), 2)) 
    }

generate_driver_udf = udf(generate_driver_raw, driver_schema)

raw_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

drivers_stream = raw_stream.withColumn("driver_data", generate_driver_udf()) \
    .select(
        "driver_data.id",
        "driver_data.name",
        "driver_data.car_number",
        "driver_data.experience",
        "driver_data.rating",
        current_timestamp().alias("ingested_at")
    )


user_email = spark.sql("SELECT current_user()").collect()[0][0]
checkpoint_dir = f"/Workspace/Users/{user_email}/checkpoints/bronze_drivers"

spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse_db")


query_bronze = drivers_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(availableNow=True) \
    .toTable("lakehouse_db.bronze_drivers")

query_bronze.awaitTermination()

print("Поток Bronze слоя успешно выполнен и данные записаны!")

In [0]:
import random
from pyspark.sql.functions import udf, current_timestamp
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField
from faker import Faker

driver_schema_v2 = StructType([
    StructField("id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("car_number", StringType(), True),
    StructField("experience", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("phone", StringType(), True)
])

def generate_driver_raw_v2():
    fake_worker = Faker()
    return {
        "id": fake_worker.uuid4(),
        "name": fake_worker.name(),
        "car_number": fake_worker.license_plate(),
        "experience": int(fake_worker.random_int(min=1, max=20)),
        "rating": float(round(random.uniform(3.5, 5.0), 2)),
        "phone": fake_worker.phone_number()  
    }

generate_driver_udf_v2 = udf(generate_driver_raw_v2, driver_schema_v2)

raw_stream_v2 = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

drivers_stream_v2 = raw_stream_v2.withColumn("driver_data", generate_driver_udf_v2()) \
    .select(
        "driver_data.id",
        "driver_data.name",
        "driver_data.car_number",
        "driver_data.experience",
        "driver_data.rating",
        "driver_data.phone",
        current_timestamp().alias("ingested_at")
    )

user_email = spark.sql("SELECT current_user()").collect()[0][0]
checkpoint_dir_v2 = f"/Workspace/Users/{user_email}/checkpoints/bronze_drivers_v2"


query_bronze_v2 = drivers_stream_v2.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_dir_v2) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable("lakehouse_db.bronze_drivers")

query_bronze_v2.awaitTermination()

print("Схема успешно обновлена! Столбец 'phone' добавлен в lakehouse_db.bronze_drivers.")

In [0]:
display(spark.sql("SELECT * FROM lakehouse_db.bronze_drivers LIMIT 10"))